# Data Analysis - Physics Augmentation vs GAN Fidelity

Thorough visual + metric comparison of the three synthetic voltammogram
sources used in this project:

* **Physics-Augmented** - `augmentation_module.py` (refactor of
  `full_range_data_augmentation.ipynb`)
* **TimeGAN** - `timegan_training.py`
* **WGAN-GP** - `wgangp_training.py`

against the **Real** calibration replicates, using the plotting functions in
`fidelity_atlas.py` and the metrics in `validation_metrics.py`. Every figure
uses the shared `plot_style.apply_default_plotly_layout`.

## 0 - Imports and Data Loading

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from validation_metrics import ValidationGate, PDFOverlapScore, VoltammogramFidelityIndex, quick_compare
import fidelity_atlas as fa
from augmentation_module import FullRangeDataAugmentor, AugmentationConfig

pd.set_option('display.width', 120)

In [2]:
# Potential grid (shared across every source)
E = pd.read_csv('raw/raw_potential_grid.csv').values.flatten()

# Real calibration replicates
df_real = pd.read_csv('raw/raw_signals_real.csv')
y_real = df_real['concentration'].values
X_real = df_real.drop(columns=['concentration']).values
real = (X_real, y_real)

# Synthetic sources: physics augmentation + both trained GANs
SOURCE_FILES = {
    'Physics-Augmented': 'raw/raw_signals_augmented.csv',
    'TimeGAN':           'gan_output/samples_trained_on_real/timegan_signals.csv',
    'WGAN-GP':           'gan_output/samples_trained_on_real/wgangp_signals.csv',
}

sources = {}
for name, path in SOURCE_FILES.items():
    df = pd.read_csv(path)
    sources[name] = (df.drop(columns=['concentration']).values, df['concentration'].values)
    print(f'{name:20s}  n={len(df):4d}  conc range {df.concentration.min():.3f}-{df.concentration.max():.3f} uM')

print(f'\nReal                  n={len(df_real):4d}  conc range {y_real.min():.3f}-{y_real.max():.3f} uM')

Physics-Augmented     n= 300  conc range 0.102-98.457 uM
TimeGAN               n= 300  conc range 0.104-93.360 uM
WGAN-GP               n= 300  conc range 0.104-93.360 uM

Real                  n=  40  conc range 0.100-100.000 uM


## 1 - Regenerating the physics-augmented batch through the refactored module

Demonstrates the `augmentation_module.py` API end to end: build a
`FullRangeDataAugmentor` from the calibration workbook, tune the pipeline via
`AugmentationConfig`, and generate a fresh batch. This is the same call a
Streamlit frontend would make behind a "Regenerate" button.

In [3]:
CONCENTRATIONS = [
    100, 100, 100,  50,  50,  50,  25,  25,  25,  15,  15,  15,
     10,  10,  10, 7.5, 7.5, 7.5,   5,   5,   5, 2.5, 2.5,
      1,   1,   1, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5,
   0.25, 0.25, 0.25, 0.1, 0.1, 0.1, 0.1, 0.1,
]

augmentor = FullRangeDataAugmentor.from_excel(
    'datasets/Standard calibration in culture media_extended.xlsx',
    concentrations_uM=CONCENTRATIONS)

config = AugmentationConfig(n_total_synthetic=300, low_conc_threshold_uM=2.5, low_conc_boost=2.0)
fresh_dataset = augmentor.generate(config)

print(f'Regenerated {len(fresh_dataset)} signals via FullRangeDataAugmentor.generate()')
fresh_dataset.to_raw_dataframe().head(3)

Regenerated 300 signals via FullRangeDataAugmentor.generate()


,concentration,I_0,I_1,I_2,I_3,I_4,I_5,I_6,I_7,I_8,...,I_219,I_220,I_221,I_222,I_223,I_224,I_225,I_226,I_227,I_228
0,0.101904,0.084463,0.095319,0.014674,-0.049810,-0.001205,0.021260,0.052182,0.016707,-0.031395,...,1.057789,1.106057,1.165211,1.161524,1.197339,1.168215,0.992325,0.607369,0.165792,-0.038278
1,0.103201,-0.132588,-0.132588,-0.003258,0.067439,0.087510,0.053263,-0.029209,-0.138808,-0.047243,...,0.831299,1.183813,0.994181,1.050627,1.192267,1.361844,1.172963,1.141421,0.900058,0.429325
2,0.104348,0.020633,0.020535,-0.030306,0.052442,0.046828,0.108840,-0.020615,-0.070460,-0.010245,...,1.028466,1.143109,1.256019,1.259899,1.287209,1.169676,0.881896,0.491334,0.137231,0.024316


## 2 - Simple overlay: real vs every synthetic source

The most direct sanity check - do the synthetic curves sit on top of the
real replicates?

In [4]:
fa.plot_signal_overlay(E, real, sources, concentration=10.0,
                       title='Signal overlay near c = 10 uM').show()

In [5]:
fa.plot_overlay_grid(E, real, sources, concentrations=[0.5, 10.0, 75.0]).show()

## 3 - Mean ± std envelope

A second point of view on the same question: instead of individual noisy
traces, compare the *average shape* and *spread* per source.

In [6]:
for c in [0.5, 10.0, 75.0]:
    fa.plot_mean_envelope_comparison(E, real, sources, concentration=c).show()

## 4 - Feature-space comparisons

### 4.1 Peak current (Ip) vs concentration

Checks the dose-response curve each source implicitly learned, against the
PCHIP spline fit to the real anchors.

In [7]:
fa.plot_ip_vs_concentration(E, real, sources, ip_spline=augmentor.ip_spline).show()

### 4.2 PCA of peak-window shapes

Projects the Ip-normalised peak window of every signal into 2D. Tight
overlap with the black Real stars means the source is not inventing new peak
shapes.

In [8]:
fa.plot_feature_pca_scatter(E, real, sources).show()

## 5 - Run the validation gate once per source

Everything below reuses these `gate_results` - no metric is recomputed by
the plotting layer, it is only visualised.

In [ ]:
gate = ValidationGate(E, X_real, y_real)
gate_results = {name: gate.run(X, y, verbose=False, run_tiers=[1, 2])
                for name, (X, y) in sources.items()}

for name, r in gate_results.items():
    print(f"{name:20s}  JSD={r['mean_jsd']:.4f}  MMD2={r['mmd2']:.6f}  "
          f"SWD={r['swd_mean']:.4f}  delta_ACF={r['delta_acf']:.5f}  VFI={r['vfi']:.4f}")\
        


Physics-Augmented     JSD=0.0749  MMD2=0.019505  SWD=0.1495  delta_ACF=0.09473  VFI=0.8669
TimeGAN               JSD=0.1300  MMD2=0.192612  SWD=0.2928  delta_ACF=0.51084  VFI=0.5166
WGAN-GP               JSD=0.0593  MMD2=0.063509  SWD=0.1379  delta_ACF=0.01733  VFI=0.8849


## 6 - What does the PDF Overlap Score mean?

`PDFOverlapScore` is the Bhattacharyya intersection between the real and
synthetic Ip-normalised peak-current histograms. The explainer plot below
shades exactly that intersection region for one concentration and one
source, then the bar chart shows the same score across every concentration
class and every source.

In [9]:
pdf_scorer = PDFOverlapScore()
pdf_overlap_by_source = {}
for name, (X, y) in sources.items():
    overlaps = pdf_scorer.per_class(X_real, y_real, X, y, E)
    overlaps['mean'] = pdf_scorer.mean(overlaps)
    pdf_overlap_by_source[name] = overlaps

for name, source_batch in sources.items():
    fa.plot_pdf_overlap_explainer(E, real, source_batch, concentration=10.0, source_name=name).show()

In [10]:
fa.plot_pdf_overlap_by_class(pdf_overlap_by_source).show()

## 7 - What does the VoltammogramFidelityIndex mean?

VFI starts at 1.0 and subtracts four normalised penalties (JSD, SWD, ACF,
feature-Wasserstein). `compute_vfi_components` re-applies the same clipping
formulas as `VoltammogramFidelityIndex.compute` to expose the four terms
(the class itself only returns the final scalar), so we can see *which*
penalty is driving each source's score down.

In [13]:
vfi_components = {name: fa.compute_vfi_components(r) for name, r in gate_results.items()}
pd.DataFrame(vfi_components).T

,p_jsd,p_swd,p_acf,p_feat,vfi
Physics-Augmented,0.074913,0.149541,0.094734,0.213206,0.866901
TimeGAN,0.130022,0.292765,0.510839,1.000000,0.516594
WGAN-GP,0.059300,0.137868,0.017331,0.245713,0.884947


In [14]:
fa.plot_vfi_breakdown_bars(vfi_components).show()

In [15]:
fa.plot_fidelity_radar(vfi_components).show()

## 8 - Complementing MMD / SWD / JSD from the validation gate

Per-class JSD (where is the distributional mismatch concentrated?) and a
side-by-side bar panel of the gate's headline scalar metrics.

In [16]:
jsd_maps_by_source = {name: r['jsd_map'] for name, r in gate_results.items()}
fa.plot_jsd_per_class(jsd_maps_by_source).show()

In [17]:
comparison_df = quick_compare(E, X_real, y_real, sources, run_tiers=[1, 2])
comparison_df

,batch,KS_mean_frac,JSD_mean,MMD2,SWD_mean,PFF_class_frac,RS_R2_synth,delta_ACF,VFI,VFI_label,PDF_overlap_mean,tier1_pass,tier2_pass
0,Physics-Augmented,0.9624,0.0749,0.019505,0.1495,0.833,0.9795,0.09473,0.8669,Good,0.7072,True,True
1,TimeGAN,0.3118,0.1300,0.192612,0.2928,0.000,0.9523,0.51084,0.5166,Poor,0.6625,False,False
2,WGAN-GP,0.6882,0.0593,0.063509,0.1379,0.667,0.9538,0.01733,0.8849,Good,0.7610,False,False


In [18]:
fa.plot_metric_summary_bars(comparison_df, metrics=('JSD_mean', 'MMD2', 'SWD_mean')).show()

## 9 - Verdict

Final side-by-side table: gate pass/fail, VFI and PDF Overlap mean per
source.

In [19]:
verdict_rows = []
for name, r in gate_results.items():
    verdict_rows.append({
        'source': name,
        'tier1_pass': r.get('tier1_pass'),
        'tier2_pass': r.get('tier2_pass'),
        'VFI': round(r['vfi'], 4),
        'PDF_overlap_mean': round(pdf_overlap_by_source[name]['mean'], 4),
    })
pd.DataFrame(verdict_rows).sort_values('VFI', ascending=False)

,source,tier1_pass,tier2_pass,VFI,PDF_overlap_mean
2,WGAN-GP,False,False,0.8849,0.7610
0,Physics-Augmented,True,True,0.8669,0.7072
1,TimeGAN,False,False,0.5166,0.6625
